
## Organic Food Supply-Chain Optimization

Install amplpy, pandas and other packages.

In [ ]:
!pip install -q amplpy ampltools

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 31.0 MB/s eta 0:00:00


Setup AMPL and select solvers.

In [ ]:
# Google Colab & AMPL integration
MODULES, LICENSE_UUID = ["coin", 'gurobi', "cplex", "highs", "gokestrel"], "42fc7eb6-69aa-445d-b655-3ad24d836541"
from amplpy import tools
from ampltools import cloud_platform_name, ampl_notebook, register_magics

# instantiate AMPL object and register magics
if cloud_platform_name() is None:
    ampl = AMPL() # Use local installation of AMPL
else:
    ampl = tools.ampl_notebook(modules=MODULES, license_uuid=LICENSE_UUID, g=globals())

register_magics(ampl_object=ampl)

Licensed to Bundle #6741.7193 expiring 20241231: INFO 645 Prescriptive Analytics, Prof. Paul Brooks, Virginia Commonwealth University.


Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# Read data

In [ ]:
import pandas as pd
from amplpy import AMPL

# Load the Excel file
excel_file_path = '/content/drive/MyDrive/645 - PRESCRIPTIVE ANALYSIS/OrganicFoodSupply.xlsx'
preparation_data = pd.read_excel(excel_file_path, sheet_name='Sheet1')
shipping_data = pd.read_excel(excel_file_path, sheet_name='Sheet2')

In [ ]:
# Process the preparation data
preparation_centers = preparation_data[['Preparation Center', 'Transportation Cost ($/pound) (Orchard to Preparation Center)',
                                        'Preparation Cost ($/pound)', 'Monthly Capacity (pounds)']]
preparation_centers.set_index('Preparation Center', inplace=True)

# Process the shipping data
shipping_data.set_index('Preparation Center', inplace=True)
shipping_costs = shipping_data

# Extract store names
stores = shipping_costs.columns.tolist()

# Convert preparation data into dictionaries for AMPL
transportation_cost = preparation_centers['Transportation Cost ($/pound) (Orchard to Preparation Center)'].to_dict()
preparation_cost = preparation_centers['Preparation Cost ($/pound)'].to_dict()
monthly_capacity = preparation_centers['Monthly Capacity (pounds)'].to_dict()

# Convert shipping costs to a two-dimensional dictionary for AMPL
shipping_cost_dict = {}
for prep_center in shipping_costs.index:
    for store in stores:
        shipping_cost_dict[(prep_center, store)] = shipping_costs.loc[prep_center, store]

# Print out the prepared dictionaries to verify
print("Transportation Costs:", transportation_cost)
print("\nPreparation Costs:", preparation_cost)
print("\nMonthly Capacities:", monthly_capacity)
print("\nShipping Costs:", shipping_cost_dict)

# Create a sample monthly demand for the stores
monthly_demand = pd.Series([300, 500, 400, 200], index=stores)
monthly_demand_dict = monthly_demand.to_dict()

# Print to verify monthly demand
print("\nMonthly Demand:", monthly_demand_dict)


Transportation Costs: {1: 0.45, 2: 1.0, 3: 1.62}

Preparation Costs: {1: 0.15, 2: 0.2, 3: 0.18}

Monthly Capacities: {1: 300, 2: 500, 3: 800}

Shipping Costs: {(1, 'Organic Orchard'): 0.8, (1, 'Fresh & Local'): 1.1, (1, 'Healthy Pantry'): 0.7, (1, "Season's Harvest"): 1.4, (2, 'Organic Orchard'): 1.2, (2, 'Fresh & Local'): 1.1, (2, 'Healthy Pantry'): 0.5, (2, "Season's Harvest"): 1.4, (3, 'Organic Orchard'): 0.2, (3, 'Fresh & Local'): 1.4, (3, 'Healthy Pantry'): 1.3, (3, "Season's Harvest"): 1.7}

Monthly Demand: {'Organic Orchard': 300, 'Fresh & Local': 500, 'Healthy Pantry': 400, "Season's Harvest": 200}



Define model.

In [ ]:
# Define the AMPL model
ampl.eval('''
reset;

# Sets
set P;  # Preparation Centers
set S;  # Stores

# Parameters
param transportation_cost {i in P};  # Transportation cost from orchard to preparation center
param preparation_cost {i in P};     # Preparation cost per pound at preparation center
param monthly_capacity {i in P};      # Monthly capacity at each preparation center
param shipping_cost {i in P, j in S}; # Shipping cost from preparation center to store
param demand {j in S};                 # Monthly demand for each store

# Decision Variables
var x {i in P, j in S} >= 0; # Amount of apples prepared and shipped from preparation center i to store j

# Objective Function
minimize total_cost: sum{i in P, j in S} (transportation_cost[i] + preparation_cost[i] + shipping_cost[i,j]) * x[i,j];

# Constraints
subject to capacity_constraint {i in P}:
    sum{j in S} x[i,j] <= monthly_capacity[i]; # Capacity at each preparation center

subject to demand_constraint {j in S}:
    sum{i in P} x[i,j] = demand[j]; # Demand at each store
''')


Provide data to the model.

In [ ]:
ampl.set['P'] = preparation_centers.index.tolist()
ampl.set['S'] = stores

ampl.param['transportation_cost'] = transportation_cost  # Transportation costs
ampl.param['preparation_cost'] = preparation_cost        # Preparation costs
ampl.param['monthly_capacity'] = monthly_capacity        # Monthly capacity
ampl.param['shipping_cost'] = shipping_cost_dict         # Shipping costs
ampl.param['demand'] = monthly_demand_dict               # Monthly demand


Display problem formulation.

In [ ]:
ampl.eval('''expand;''')

minimize total_cost:
	1.4*x[1,'Organic Orchard'] + 1.7*x[1,'Fresh & Local'] + 
	1.3*x[1,'Healthy Pantry'] + 2*x[1,"Season's Harvest"] + 
	2.4*x[2,'Organic Orchard'] + 2.3*x[2,'Fresh & Local'] + 
	1.7*x[2,'Healthy Pantry'] + 2.6*x[2,"Season's Harvest"] + 
	2*x[3,'Organic Orchard'] + 3.2*x[3,'Fresh & Local'] + 
	3.1*x[3,'Healthy Pantry'] + 3.5*x[3,"Season's Harvest"];

subject to capacity_constraint[1]:
	x[1,'Organic Orchard'] + x[1,'Fresh & Local'] + x[1,'Healthy Pantry']
	 + x[1,"Season's Harvest"] <= 300;

subject to capacity_constraint[2]:
	x[2,'Organic Orchard'] + x[2,'Fresh & Local'] + x[2,'Healthy Pantry']
	 + x[2,"Season's Harvest"] <= 500;

subject to capacity_constraint[3]:
	x[3,'Organic Orchard'] + x[3,'Fresh & Local'] + x[3,'Healthy Pantry']
	 + x[3,"Season's Harvest"] <= 800;

subject to demand_constraint['Organic Orchard']:
	x[1,'Organic Orchard'] + x[2,'Organic Orchard'] + 
	x[3,'Organic Orchard'] = 300;

subject to demand_constraint['Fresh & Local']:
	x[1,'Fresh & Local']

Set solver and solve.

In [ ]:
ampl.setOption('solver', 'cplex')
ampl.solve()

CPLEX 22.1.1:  - Version identifier: 22.1.1.0 | 2022-11-28 | 9160aff4d
 - CPXPARAM_Simplex_Display                         0
 - CPXPARAM_MIP_Display                             0
 - CPXPARAM_Barrier_Display                         0
CPLEX 22.1.1: optimal solution; objective 3040
4 simplex iterations


Print solution and results.

In [ ]:
# Solve the model
ampl.solve()

In [ ]:
# Retrieve the decision variables and  total cost (objective function value)
x = ampl.get_variable('x').get_values()
total_cost = ampl.get_objective('total_cost').value()

# Display the results
print("Optimal Solution (Amount of Apples Shipped from Each Preparation Center to Each Store):\n")
print(f"{'Preparation Center':<20} {'Store':<20} {'Amount Shipped (x.val)':<25}")
print("="*65)

print(x)

print("\nTotal Cost of Transportation, Preparation, and Shipping:")
print(f"${total_cost:,.2f}")


Optimal Solution (Amount of Apples Shipped from Each Preparation Center to Each Store):

Preparation Center   Store                Amount Shipped (x.val)   
   index0       index1    |    x.val    
     1         'Fresh & Local' |     100     
     1         'Healthy Pantry' |      0      
     1         'Organic Orchard' |      0      
     1         'Season''s Harvest' |     200     
     2         'Fresh & Local' |     100     
     2         'Healthy Pantry' |     400     
     2         'Organic Orchard' |      0      
     2         'Season''s Harvest' |      0      
     3         'Fresh & Local' |     300     
     3         'Healthy Pantry' |      0      
     3         'Organic Orchard' |     300     
     3         'Season''s Harvest' |      0      


Total Cost of Transportation, Preparation, and Shipping:
$3,040.00


The optimal solution - total cost of transportation, preparation, and shipping is \$3,040.
